# aa2codon v2: Prediction and analysis
Select a GPU runtime when available. Run training first or provide existing v2 checkpoints.

In [ ]:
!git clone -b pytorch-xielu-uncertainty https://github.com/pyypyyy/aa2codon.git /content/aa2codon_v2
%cd /content/aa2codon_v2/pytorch_v2
!pip -q install biopython captum plotly
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, torch
from pathlib import Path
from model import load_model, ALLOWED, AA_ID, CODONS
from uncertainty import predict
from visualization import uncertainty_plot, codon_detail, attribution_plot, calibration_plot, cross_attention_plot
from interpretability import explain, attention_maps
OUT = Path('/content/drive/MyDrive/aa2codon_v2/models')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
manifest = json.loads((OUT/'manifest.json').read_text())
models = [load_model(OUT/r['path'], device)[0] for r in manifest]
temps = [r['temperature'] for r in manifest]

In [ ]:
PROTEIN = 'MKAVGLPSGKTA*'
result = predict(models, PROTEIN, temps)
print(result['dna'])
uncertainty_plot(result).show()
calibration_plot(json.loads((OUT/'ensemble_test.json').read_text())).show()

In [ ]:
from ipywidgets import interact, IntSlider
@interact(position=IntSlider(min=1,max=len(result['codons']),value=1))
def inspect(position):
    codon_detail(result,position).show()

In [ ]:
POSITION = 6 # one-based, choose a degenerate residue
ids = ALLOWED[AA_ID[result['protein'][POSITION-1]]].nonzero().flatten().tolist()
chosen = result['codons'][POSITION-1]
alts = sorted([j for j in ids if CODONS[j]!=chosen],key=lambda j:float(result['mean'][POSITION-1,j]),reverse=True)
assert alts, 'Choose a residue with multiple synonymous codons.'
ALT = CODONS[alts[0]]
attrs = [explain(m,result['protein'],result['codons'],POSITION-1,ALT,temperature=t) for m,t in zip(models,temps)]
print('Integrated gradients completeness residuals:', [r['convergence_delta'] for r in attrs])
attribution_plot(attrs,result['protein'],result['codons'],POSITION,ALT).show()

In [ ]:
weights = attention_maps(models[0],result['protein'],result['codons'])
cross_attention_plot(weights,result['protein'],result['codons'],layer=-1,head=0).show()